# Aliasing demo

This notebook demonstrates the aliasing capabilities of the ACCESS-NRI Intake catalog.
Aliasing lets you search the catalog using familiar CMIP-style vocabulary (field names,
variable codes, frequency strings, etc.) and have those terms automatically mapped to
the underlying canonical values stored in the catalog.

For a full reference, see the [Aliasing](aliases.rst) documentation page.

> **Note**: This notebook must be run on Gadi with access to the `xp65` project
> and the relevant data projects. See the [How do I use it?](how.rst) page for prerequisites.

## Contents

1. [Setup](#setup)
2. [Field aliasing — CMIP column names](#field-aliasing)
3. [Value aliasing — frequency](#value-aliasing-frequency)
4. [Value aliasing — realm](#value-aliasing-realm)
5. [Value aliasing — model shortcuts](#value-aliasing-model)
6. [Value aliasing — experiment shortcuts](#value-aliasing-experiment)
7. [CMIP variable search (scalar)](#cmip-variable-scalar)
8. [CMIP variable search (list)](#cmip-variable-list)
9. [Regex passthrough](#regex-passthrough)
10. [Controlling alias warnings](#warnings)
11. [Escaping aliasing with .unwrap()](#unwrap)

<a name="setup"></a>
## 1. Setup

Load the catalog. On Gadi this uses the default catalog location in `/g/data/xp65`.

In [1]:
import warnings
import intake

cat = intake.cat.access_nri
cat

/home/189/ct1163/access-nri-intake-catalog/bin/venv/lib/python3.11/site-packages/fastprogress/fastprogress.py:107: UserWarning: Couldn't import ipywidgets properly, progress bar will use console behavior
  warn("Couldn't import ipywidgets properly, progress bar will use console behavior")


,model,description,realm,frequency,variable
name,,,,,
01deg_jra55_ryf_Control,{ACCESS-OM2-01},"{0.1° ACCESS-OM2 repeat year forcing control run for the simulations performed in Huguenin et al. (2024, GRL)}","{seaIce, ocean}","{fx, 1mon}","{total_ocean_fprec_melt_heat, congel_m, dxt, temp_tendency, tx_trans_nrho_submeso, potrho, grid_xt_ocean, strairx_m, total_ocean_pme_river, pe_tot, ty_trans_rho, uocn_m, ty_trans, neutralrho_edges..."
01deg_jra55_ryf_ENFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 El Níño run for the simulations performed in Huguenin et al. (2024, GRL)}","{seaIce, ocean}","{fx, 1mon}","{total_ocean_fprec_melt_heat, congel_m, dxt, temp_tendency, grid_xt_ocean, potrho, total_ocean_pme_river, strairx_m, pe_tot, ty_trans_rho, uocn_m, neutralrho_edges, ty_trans, yt_ocean, hs_m, ANGLE..."
01deg_jra55_ryf_LNFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 La Níña run for the simulations performed in Huguenin et al. (2024, GRL)}","{seaIce, ocean}","{fx, 1mon}","{total_ocean_fprec_melt_heat, congel_m, dxt, temp_tendency, potrho, grid_xt_ocean, strairx_m, total_ocean_pme_river, pe_tot, ty_trans_rho, uocn_m, ty_trans, neutralrho_edges, yt_ocean, hs_m, ANGLE..."
01deg_jra55v13_ryf9091,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991)},"{seaIce, ocean}","{1day, fx, 3mon, 3hr, 1mon}","{passive_prydz, total_ocean_fprec_melt_heat, congel_m, dxt, fprec, temp_tendency, potrho, grid_xt_ocean, total_ocean_pme_river, strairx_m, pe_tot, pot_rho_1, uhrho_et, ty_trans_rho, xu_ocean_sub02..."
01deg_jra55v13_ryf9091_easterlies_down10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica decreased by 10%.},"{seaIce, ocean}","{1day, fx, 1mon}","{total_ocean_fprec_melt_heat, congel_m, dxt, fprec, potrho, grid_xt_ocean, total_ocean_pme_river, strairx_m, pe_tot, ty_trans_rho, uocn_m, ty_trans, neutralrho_edges, yt_ocean, hs_m, ANGLET, vert_..."
01deg_jra55v13_ryf9091_easterlies_up10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica increased by 10%.},"{seaIce, ocean}","{fx, 1day, 1mon}","{total_ocean_fprec_melt_heat, congel_m, dxt, fprec, potrho, grid_xt_ocean, total_ocean_pme_river, strairx_m, pe_tot, ty_trans_rho, uocn_m, ty_trans, neutralrho_edges, yt_ocean, hs_m, ANGLET, vert_..."
01deg_jra55v13_ryf9091_easterlies_up10_meridional,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and meridional wind speed around Antarctica increased by 10%.},"{seaIce, ocean}","{1day, fx, 1mon}","{total_ocean_fprec_melt_heat, congel_m, dxt, fprec, potrho, grid_xt_ocean, strairx_m, total_ocean_pme_river, pe_tot, ty_trans_rho, uocn_m, ty_trans, neutralrho_edges, yt_ocean, hs_m, ANGLET, vert_..."
01deg_jra55v13_ryf9091_easterlies_up10_zonal,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal wind speed around Antarctica increased by 10%.},"{seaIce, ocean}","{fx, 1day, 1mon}","{total_ocean_fprec_melt_heat, congel_m, dxt, fprec, potrho, grid_xt_ocean, strairx_m, total_ocean_pme_river, pe_tot, ty_trans_rho, uocn_m, ty_trans, neutralrho_edges, yt_ocean, hs_m, ANGLET, vert_..."
01deg_jra55v13_ryf9091_qian_wthmp,{ACCESS-OM2},"{Future perturbations with wind, thermal and meltwater forcing, branching off 01deg_jra55v13_ryf9091, as described in Li et al. 2023, https://www.nature.com/articles/s41586-023-05762-w}","{seaIce, ocean}","{fx, 1mon}","{total_ocean_fprec_melt_heat, congel_m, dxt, fprec, temp_tendency, potrho, grid_xt_ocean, total_ocean_pme_river, strairx_m, pe_tot, ty_trans_rho, uocn_m, ty_trans, yt_ocean, hs_m, ANGLET, temp_vdi..."


<a name="field-aliasing"></a>
## 2. Field aliasing — CMIP Style Variable ID's

The top-level catalog accepts CMIP-style names in `.search()`. 

Each call will emit a `UserWarning` showing the mapping that was applied.

In [2]:
# Using 'variable_id' — maps to the catalog's 'variable' column
results = cat.search(variable="tas")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198199/494749647.py:2: UserWarning: Value aliasing: variable='tas' → variable=['fld_s03i236','tas']
  results = cat.search(variable="tas")


,model,description,realm,frequency,variable
name,,,,,
AUS2200,{AUS2200},"{""AUS2200 AMIP simulations collection. Each simulation is a limited area model study of the entire Australian continent at 2.2 km resolution, using the UM atmospheric model. The simulations cover ...",{atmos},"{1hr, subhr}",{tas}
HI_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP}",{atmos},"{1day, 3hr, 1mon}",{fld_s03i236}
HI_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP}",{atmos},"{1day, 1mon}",{fld_s03i236}
HI_nl_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP, and land-use change disabled}",{atmos},"{1day, 1mon}",{fld_s03i236}
HI_noluc_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP, and land-use change disabled}",{atmos},"{1day, 3hr, 1mon}",{fld_s03i236}
PI_GWL_B2035,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2035 }",{atmos},"{1day, 1mon}",{fld_s03i236}
PI_GWL_B2040,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2040}",{atmos},"{1day, 1mon}",{fld_s03i236}
PI_GWL_B2045,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2045}",{atmos},"{1day, 1mon}",{fld_s03i236}
PI_GWL_B2050,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2050}",{atmos},"{1day, 1mon}",{fld_s03i236}


<a name="value-aliasing-frequency"></a>
## 3. Value aliasing — frequency

Human-readable frequency strings are mapped to the catalog's stored values
(e.g. `"daily"` → `"1day"`). The search expands to include **both** terms, so
any data already labelled `"daily"` is also included.

In [3]:
# 'daily' expands to ['1day', 'daily']
results = cat.search(frequency="daily")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198199/327739463.py:2: UserWarning: Value aliasing: frequency='daily' → frequency=['1day','daily']
  results = cat.search(frequency="daily")


,model,description,realm,frequency,variable
name,,,,,
01deg_jra55v13_ryf9091,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991)},{ocean},{1day},"{sw_edges_ocean, tau_x, vhrho_nt, pme_river, average_T2, xt_ocean, time, surface_salt, tau_y, st_edges_ocean, pot_rho_1, uhrho_et, surface_temp, yu_ocean, yt_ocean, sfc_hflux_pme, sfc_hflux_from_r..."
01deg_jra55v13_ryf9091_easterlies_down10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica decreased by 10%.},{ocean},{1day},"{tau_x, pme_river, xt_ocean, average_T2, time, surface_salt, tau_y, surface_temp, yu_ocean, yt_ocean, sfc_hflux_pme, sfc_hflux_from_runoff, frazil_3d_int_z, time_bounds, vsurf, eta_t, nv, xu_ocean..."
01deg_jra55v13_ryf9091_easterlies_up10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica increased by 10%.},{ocean},{1day},"{tau_x, pme_river, xt_ocean, average_T2, time, surface_salt, tau_y, surface_temp, yu_ocean, yt_ocean, sfc_hflux_pme, sfc_hflux_from_runoff, frazil_3d_int_z, time_bounds, vsurf, eta_t, nv, xu_ocean..."
01deg_jra55v13_ryf9091_easterlies_up10_meridional,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and meridional wind speed around Antarctica increased by 10%.},{ocean},{1day},"{tau_x, pme_river, xt_ocean, average_T2, time, surface_salt, tau_y, surface_temp, yu_ocean, yt_ocean, sfc_hflux_pme, sfc_hflux_from_runoff, frazil_3d_int_z, time_bounds, vsurf, eta_t, nv, xu_ocean..."
01deg_jra55v13_ryf9091_easterlies_up10_zonal,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal wind speed around Antarctica increased by 10%.},{ocean},{1day},"{tau_x, pme_river, xt_ocean, average_T2, time, surface_salt, tau_y, surface_temp, yu_ocean, yt_ocean, sfc_hflux_pme, sfc_hflux_from_runoff, frazil_3d_int_z, time_bounds, vsurf, eta_t, nv, xu_ocean..."
01deg_jra55v13_ryf9091_weddell_down2,{ACCESS-OM2-01},"{Weddell Sea decreased meltwater perturbation experiment, branched off 01deg_jra55v13_ryf9091. }",{ocean},{1day},"{tau_x, pme_river, xt_ocean, average_T2, time, surface_salt, tau_y, surface_temp, yu_ocean, yt_ocean, sfc_hflux_pme, sfc_hflux_from_runoff, frazil_3d_int_z, time_bounds, vsurf, eta_t, nv, xu_ocean..."
01deg_jra55v13_ryf9091_weddell_up1,{ACCESS-OM2-01},"{Weddell Sea increased meltwater perturbation experiment, branched off 01deg_jra55v13_ryf9091. }",{ocean},{1day},"{tau_x, pme_river, average_T2, xt_ocean, time, surface_salt, tau_y, surface_temp, yu_ocean, yt_ocean, sfc_hflux_pme, sfc_hflux_from_runoff, frazil_3d_int_z, time_bounds, vsurf, eta_t, nv, xu_ocean..."
01deg_jra55v140_iaf,{ACCESS-OM2-01},{Cycle 1 of 0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.4.0 OMIP2 interannual forcing},"{seaIce, ocean}",{1day},"{scalar_axis, aicen, total_ocean_fprec_melt_heat, total_ocean_mh_flux, dxt, bottom_temp, total_ocean_pme_river, pe_tot, uvel, yt_ocean, ANGLET, total_net_sfc_heating, congel, salt, average_T1, tem..."
01deg_jra55v140_iaf_cycle2,{ACCESS-OM2-01},{Cycle 2 of 0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.4.0 OMIP2 interannual forcing},"{seaIce, ocean}",{1day},"{aicen, surface_pot_temp_min, total_ocean_fprec_melt_heat, total_ocean_mh_flux, dxt, bottom_temp, total_ocean_pme_river, pe_tot, fsurf_ai, uvel, bottom_temp_max, yt_ocean, surface_pot_temp, ANGLET..."


In [4]:
# 'monthly' expands to ['1mon', 'monthly']
results = cat.search(frequency="monthly")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198199/4282274679.py:2: UserWarning: Value aliasing: frequency='monthly' → frequency=['1mon','monthly']
  results = cat.search(frequency="monthly")


,model,description,realm,frequency,variable
name,,,,,
01deg_jra55_ryf_Control,{ACCESS-OM2-01},"{0.1° ACCESS-OM2 repeat year forcing control run for the simulations performed in Huguenin et al. (2024, GRL)}","{seaIce, ocean}",{1mon},"{total_ocean_fprec_melt_heat, congel_m, dxt, temp_tendency, tx_trans_nrho_submeso, potrho, grid_xt_ocean, strairx_m, total_ocean_pme_river, pe_tot, ty_trans_rho, uocn_m, ty_trans, neutralrho_edges..."
01deg_jra55_ryf_ENFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 El Níño run for the simulations performed in Huguenin et al. (2024, GRL)}","{seaIce, ocean}",{1mon},"{total_ocean_fprec_melt_heat, congel_m, dxt, temp_tendency, grid_xt_ocean, potrho, strairx_m, total_ocean_pme_river, pe_tot, ty_trans_rho, uocn_m, neutralrho_edges, ty_trans, yt_ocean, hs_m, ANGLE..."
01deg_jra55_ryf_LNFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 La Níña run for the simulations performed in Huguenin et al. (2024, GRL)}","{seaIce, ocean}",{1mon},"{total_ocean_fprec_melt_heat, congel_m, dxt, temp_tendency, grid_xt_ocean, potrho, strairx_m, total_ocean_pme_river, pe_tot, ty_trans_rho, uocn_m, ty_trans, neutralrho_edges, yt_ocean, hs_m, ANGLE..."
01deg_jra55v13_ryf9091,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991)},"{seaIce, ocean}",{1mon},"{passive_prydz, total_ocean_fprec_melt_heat, congel_m, dxt, fprec, temp_tendency, grid_xt_ocean, potrho, total_ocean_pme_river, strairx_m, pe_tot, ty_trans_rho, uocn_m, ty_trans, neutralrho_edges,..."
01deg_jra55v13_ryf9091_easterlies_down10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica decreased by 10%.},"{seaIce, ocean}",{1mon},"{total_ocean_fprec_melt_heat, congel_m, dxt, fprec, grid_xt_ocean, potrho, total_ocean_pme_river, strairx_m, pe_tot, ty_trans_rho, uocn_m, neutralrho_edges, ty_trans, yt_ocean, hs_m, ANGLET, vert_..."
01deg_jra55v13_ryf9091_easterlies_up10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica increased by 10%.},"{seaIce, ocean}",{1mon},"{total_ocean_fprec_melt_heat, congel_m, dxt, fprec, potrho, grid_xt_ocean, total_ocean_pme_river, strairx_m, pe_tot, ty_trans_rho, uocn_m, neutralrho_edges, ty_trans, yt_ocean, hs_m, ANGLET, vert_..."
01deg_jra55v13_ryf9091_easterlies_up10_meridional,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and meridional wind speed around Antarctica increased by 10%.},"{seaIce, ocean}",{1mon},"{total_ocean_fprec_melt_heat, congel_m, dxt, fprec, potrho, grid_xt_ocean, strairx_m, total_ocean_pme_river, pe_tot, ty_trans_rho, uocn_m, neutralrho_edges, ty_trans, yt_ocean, hs_m, ANGLET, vert_..."
01deg_jra55v13_ryf9091_easterlies_up10_zonal,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal wind speed around Antarctica increased by 10%.},"{seaIce, ocean}",{1mon},"{total_ocean_fprec_melt_heat, congel_m, dxt, fprec, grid_xt_ocean, potrho, strairx_m, total_ocean_pme_river, pe_tot, ty_trans_rho, uocn_m, ty_trans, neutralrho_edges, yt_ocean, hs_m, ANGLET, vert_..."
01deg_jra55v13_ryf9091_qian_wthmp,{ACCESS-OM2},"{Future perturbations with wind, thermal and meltwater forcing, branching off 01deg_jra55v13_ryf9091, as described in Li et al. 2023, https://www.nature.com/articles/s41586-023-05762-w}","{seaIce, ocean}",{1mon},"{total_ocean_fprec_melt_heat, congel_m, dxt, fprec, temp_tendency, potrho, grid_xt_ocean, total_ocean_pme_river, strairx_m, pe_tot, ty_trans_rho, uocn_m, ty_trans, yt_ocean, hs_m, ANGLET, temp_vdi..."


<a name="value-aliasing-realm"></a>
## 4. Value aliasing — realm

Common realm strings map to the catalog's stored values
(e.g. `"atmosphere"` → `"atmos"`, `"sea-ice"` → `"seaIce"`).

In [5]:
# 'atmosphere' expands to ['atmos', 'atmosphere']
results = cat.search(realm="atmosphere")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198199/1345224546.py:2: UserWarning: Value aliasing: realm='atmosphere' → realm=['atmos','atmosphere']
  results = cat.search(realm="atmosphere")


,model,description,realm,frequency,variable
name,,,,,
AUS2200,{AUS2200},"{""AUS2200 AMIP simulations collection. Each simulation is a limited area model study of the entire Australian continent at 2.2 km resolution, using the UM atmospheric model. The simulations cover ...",{atmos},"{1hr, subhr, 6hr, 3hr}","{va, huss, eow, psl, ta, rsdt, vas, rsdsdiff, cllow, rsds, reflmax, hus, rsut, wa, grplmxrat, clw, cli, rlut, mrsol, pralsprof, rlds, ua, theta, clmxro, pralsns, rsdsdir, rainmxrat, clhigh, evspsb..."
HI_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP}",{atmos},"{1day, 6hr, 3hr, 1mon}","{fld_s03i873, fld_s03i857, fld_s08i208, fld_s03i236_min, fld_s03i822, fld_s00i408, fld_s03i821, fld_s02i201, fld_s03i897, fld_s03i896, fld_s17i232, fld_s03i257, pseudo_level_0, fld_s03i807, fld_s0..."
HI_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP}",{atmos},"{1day, 1mon}","{fld_s03i873, fld_s03i236_min, fld_s08i208, fld_s03i822, fld_s03i857, fld_s00i408, fld_s03i821, fld_s03i897, fld_s02i201, fld_s17i232, fld_s03i896, fld_s03i257, pseudo_level_0, fld_s03i807, fld_s0..."
HI_nl_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP, and land-use change disabled}",{atmos},"{1day, 1mon}","{fld_s03i236_min, fld_s03i857, fld_s08i208, fld_s03i822, fld_s03i873, fld_s00i408, fld_s03i821, fld_s02i201, fld_s03i897, fld_s03i896, fld_s17i232, fld_s03i257, pseudo_level_0, fld_s03i807, fld_s0..."
HI_noluc_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP, and land-use change disabled}",{atmos},"{1day, 6hr, 3hr, 1mon}","{fld_s03i873, fld_s03i857, fld_s08i208, fld_s03i822, fld_s03i236_min, fld_s00i408, fld_s03i821, fld_s03i897, fld_s02i201, fld_s17i232, fld_s03i896, fld_s03i257, pseudo_level_0, fld_s03i807, fld_s0..."
PI_GWL_B2035,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2035 }",{atmos},"{1day, 1mon}","{fld_s03i236_min, fld_s03i857, fld_s08i208, fld_s03i822, fld_s03i873, fld_s00i408, fld_s03i821, fld_s02i201, fld_s17i232, fld_s03i257, pseudo_level_0, fld_s03i807, fld_s08i231, theta_level_height,..."
PI_GWL_B2040,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2040}",{atmos},"{1day, 1mon}","{fld_s03i873, fld_s03i857, fld_s08i208, fld_s03i822, fld_s03i236_min, fld_s00i408, fld_s03i821, fld_s02i201, fld_s17i232, fld_s03i257, pseudo_level_0, fld_s03i807, fld_s08i231, theta_level_height,..."
PI_GWL_B2045,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2045}",{atmos},"{1day, 1mon}","{fld_s03i873, fld_s03i857, fld_s08i208, fld_s03i236_min, fld_s03i822, fld_s00i408, fld_s03i821, fld_s02i201, fld_s17i232, fld_s03i257, pseudo_level_0, fld_s03i807, fld_s08i231, theta_level_height,..."
PI_GWL_B2050,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2050}",{atmos},"{1day, 1mon}","{fld_s03i236_min, fld_s03i857, fld_s03i873, fld_s03i822, fld_s08i208, fld_s00i408, fld_s03i821, fld_s02i201, fld_s17i232, fld_s03i257, pseudo_level_0, fld_s03i807, fld_s08i231, theta_level_height,..."


In [6]:
# 'sea-ice' expands to ['seaIce', 'sea-ice']
results = cat.search(realm="sea-ice")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198199/1193532718.py:2: UserWarning: Value aliasing: realm='sea-ice' → realm=['seaIce','sea-ice']
  results = cat.search(realm="sea-ice")


,model,description,realm,frequency,variable
name,,,,,
01deg_jra55_ryf_Control,{ACCESS-OM2-01},"{0.1° ACCESS-OM2 repeat year forcing control run for the simulations performed in Huguenin et al. (2024, GRL)}",{seaIce},{1mon},"{uatm_m, fsalt_ai_m, TLAT, tarea, TLON, congel_m, frazil_m, divu_m, dxt, mlt_onset_m, time, ANGLE, aice_m, aicen_m, strairx_m, alidf_ai_m, alvdr_ai_m, vicen_m, uocn_m, ULAT, blkmask, hi_m, strairy..."
01deg_jra55_ryf_ENFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 El Níño run for the simulations performed in Huguenin et al. (2024, GRL)}",{seaIce},{1mon},"{uatm_m, fsalt_ai_m, TLAT, tarea, TLON, congel_m, divu_m, dxt, frazil_m, mlt_onset_m, time, ANGLE, aice_m, aicen_m, strairx_m, alidf_ai_m, alvdr_ai_m, vicen_m, uocn_m, ULAT, strairy_m, blkmask, hi..."
01deg_jra55_ryf_LNFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 La Níña run for the simulations performed in Huguenin et al. (2024, GRL)}",{seaIce},{1mon},"{uatm_m, fsalt_ai_m, TLAT, tarea, TLON, congel_m, divu_m, frazil_m, dxt, mlt_onset_m, ANGLE, time, aice_m, aicen_m, strairx_m, alidf_ai_m, alvdr_ai_m, vicen_m, uocn_m, ULAT, hi_m, blkmask, strairy..."
01deg_jra55v13_ryf9091,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991)},{seaIce},{1mon},"{uatm_m, fsalt_ai_m, TLAT, tarea, TLON, congel_m, dxt, frazil_m, divu_m, mlt_onset_m, time, ANGLE, aice_m, aicen_m, strairx_m, alidf_ai_m, alvdr_ai_m, vicen_m, uocn_m, strairy_m, hi_m, ULAT, blkma..."
01deg_jra55v13_ryf9091_easterlies_down10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica decreased by 10%.},{seaIce},{1mon},"{uatm_m, fsalt_ai_m, TLAT, tarea, TLON, congel_m, frazil_m, dxt, divu_m, mlt_onset_m, ANGLE, time, aice_m, aicen_m, strairx_m, alidf_ai_m, alvdr_ai_m, vicen_m, uocn_m, strairy_m, blkmask, ULAT, hi..."
01deg_jra55v13_ryf9091_easterlies_up10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica increased by 10%.},{seaIce},{1mon},"{uatm_m, fsalt_ai_m, TLAT, tarea, TLON, congel_m, dxt, divu_m, frazil_m, mlt_onset_m, time, ANGLE, aice_m, aicen_m, strairx_m, alidf_ai_m, alvdr_ai_m, vicen_m, uocn_m, blkmask, ULAT, hi_m, strairy..."
01deg_jra55v13_ryf9091_easterlies_up10_meridional,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and meridional wind speed around Antarctica increased by 10%.},{seaIce},{1mon},"{uatm_m, fsalt_ai_m, TLAT, tarea, TLON, congel_m, divu_m, dxt, frazil_m, mlt_onset_m, ANGLE, time, aice_m, aicen_m, strairx_m, alidf_ai_m, alvdr_ai_m, vicen_m, uocn_m, strairy_m, blkmask, ULAT, hi..."
01deg_jra55v13_ryf9091_easterlies_up10_zonal,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal wind speed around Antarctica increased by 10%.},{seaIce},{1mon},"{uatm_m, fsalt_ai_m, TLAT, tarea, TLON, congel_m, dxt, frazil_m, divu_m, mlt_onset_m, time, ANGLE, aice_m, aicen_m, strairx_m, alidf_ai_m, alvdr_ai_m, vicen_m, uocn_m, strairy_m, hi_m, ULAT, blkma..."
01deg_jra55v13_ryf9091_qian_wthmp,{ACCESS-OM2},"{Future perturbations with wind, thermal and meltwater forcing, branching off 01deg_jra55v13_ryf9091, as described in Li et al. 2023, https://www.nature.com/articles/s41586-023-05762-w}",{seaIce},{1mon},"{uatm_m, fsalt_ai_m, TLAT, tarea, TLON, congel_m, divu_m, frazil_m, dxt, mlt_onset_m, ANGLE, time, aice_m, aicen_m, strairx_m, alidf_ai_m, alvdr_ai_m, vicen_m, uocn_m, strairy_m, hi_m, blkmask, UL..."


<a name="value-aliasing-model"></a>
## 5. Value aliasing — model shortcuts

Short model names map to their full catalog names.

In [7]:
# 'ACCESS-ESM1' expands to ['ACCESS-ESM1-5', 'ACCESS-ESM1']
results = cat.search(model="ACCESS-ESM1")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198199/2543793196.py:2: UserWarning: Value aliasing: model='ACCESS-ESM1' → model=['ACCESS-ESM1-5','ACCESS-ESM1']
  results = cat.search(model="ACCESS-ESM1")


,model,description,realm,frequency,variable
name,,,,,
HI_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP}","{ocean, seaIce, atmos}","{1day, 3hr, 1yr, 6hr, 1mon}","{salt_merid_flux_advect_southern, visc_crit_bih, energy_flux, bottom_salt, fld_s03i897, phycos_raw, bottom_temp, stf07, temp_merid_flux_advect_southern, fld_s03i236_max, fld_s03i806, mixdownslope_..."
HI_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP}","{ocean, seaIce, atmos}","{1mon, 1day, 1yr}","{salt_merid_flux_advect_southern, visc_crit_bih, energy_flux, bottom_salt, fld_s03i897, phycos_raw, bottom_temp, stf07, temp_merid_flux_advect_southern, fld_s03i236_max, bvfreq_bottom, fld_s03i806..."
HI_nl_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP, and land-use change disabled}","{ocean, seaIce, atmos}","{1day, 1yr, 1mon}","{salt_merid_flux_advect_southern, visc_crit_bih, energy_flux, bottom_salt, fld_s03i897, phycos_raw, bottom_temp, stf07, temp_merid_flux_advect_southern, fld_s03i236_max, fld_s03i806, mixdownslope_..."
HI_noluc_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP, and land-use change disabled}","{ocean, seaIce, atmos}","{1day, 3hr, 1mon, 6hr, 1yr}","{visc_crit_bih, salt_merid_flux_advect_southern, energy_flux, bottom_salt, fld_s03i897, phycos_raw, bottom_temp, stf07, temp_merid_flux_advect_southern, fld_s03i236_max, bvfreq_bottom, fld_s03i806..."
PI_GWL_B2035,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2035 }","{atmos, seaIce, ocean}","{1mon, 1day, 1yr}","{visc_crit_bih, salt_merid_flux_advect_southern, energy_flux, bottom_salt, phycos_raw, bottom_temp, stf07, temp_merid_flux_advect_southern, fld_s03i236_max, fld_s03i806, mixdownslope_salt, fmelttn..."
PI_GWL_B2040,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2040}","{ocean, seaIce, atmos}","{1day, 1yr, 1mon}","{salt_merid_flux_advect_southern, visc_crit_bih, energy_flux, bottom_salt, phycos_raw, bottom_temp, stf07, temp_merid_flux_advect_southern, fld_s03i236_max, fld_s03i806, mixdownslope_salt, fmelttn..."
PI_GWL_B2045,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2045}","{ocean, seaIce, atmos}","{1mon, 1day, 1yr}","{salt_merid_flux_advect_southern, visc_crit_bih, energy_flux, bottom_salt, phycos_raw, bottom_temp, stf07, temp_merid_flux_advect_southern, fld_s03i236_max, bvfreq_bottom, fld_s03i806, mixdownslop..."
PI_GWL_B2050,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2050}","{atmos, seaIce, ocean}","{1day, 1yr, 1mon}","{visc_crit_bih, salt_merid_flux_advect_southern, energy_flux, bottom_salt, phycos_raw, bottom_temp, stf07, temp_merid_flux_advect_southern, fld_s03i236_max, fld_s03i806, mixdownslope_salt, fmelttn..."
PI_GWL_B2055,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2055}","{ocean, seaIce, atmos}","{1day, 1yr, 1mon}","{salt_merid_flux_advect_southern, visc_crit_bih, energy_flux, bottom_salt, phycos_raw, bottom_temp, stf07, temp_merid_flux_advect_southern, fld_s03i236_max, fld_s03i806, mixdownslope_salt, fmelttn..."


<a name="cmip-variable-scalar"></a>
## 7. CMIP variable search — scalar

CMIP standard variable names (e.g. `tas`, `pr`, `ci`) are mapped to the raw
ACCESS field codes stored in ESM datastores. The search always includes *both*
the CMIP name and the ACCESS code, so you never miss data labelled either way.

These mappings come from `access-esm1-6-cmip-mappings.json` which covers 137
atmosphere, land, and ocean variables for ACCESS-ESM1.5 and ACCESS-ESM1.6.

In [13]:
ds = cat['HI_CN_05']
ds

,unique
filename,9435
path,9435
file_id,16
frequency,5
start_date,3705
end_date,3705
variable,655
variable_long_name,551
variable_standard_name,138
variable_cell_methods,8


In [14]:
# Search for 'tas' — CMIP name for near-surface air temperature
# Automatically expands to variable=['fld_s03i236', 'tas']
results = ds.search(variable="tas")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198199/4195522969.py:3: UserWarning: Value aliasing: variable='tas' → variable=['fld_s03i236','tas']
  results = ds.search(variable="tas")


,unique
filename,4740
path,4740
file_id,7
frequency,3
start_date,2760
end_date,2760
variable,276
variable_long_name,202
variable_standard_name,90
variable_cell_methods,5


In [15]:
# 'ci' (convection time fraction) → ['fld_s05i269', 'ci']
results = ds.search(variable="ci")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198199/3266990395.py:2: UserWarning: Value aliasing: variable='ci' → variable=['fld_s05i269','ci']
  results = ds.search(variable="ci")


,unique
filename,0
path,0
file_id,0
frequency,0
start_date,0
end_date,0
variable,0
variable_long_name,0
variable_standard_name,0
variable_cell_methods,0


<a name="cmip-variable-list"></a>
## 8. CMIP variable search — list

You can pass a list of CMIP variable names. Each is aliased individually and
the full expanded set is searched.

In [16]:
# Each CMIP name is resolved: ci→fld_s05i269, cl→fld_s02i261, tas→fld_s03i236
results = ds.search(variable=["ci", "cl", "tas"])
results

/jobfs/162823973.gadi-pbs/ipykernel_3198199/1139839834.py:2: UserWarning: Value aliasing: variable='['ci', 'cl', 'tas']' → variable=['fld_s05i269','['ci', 'cl', 'tas']']
  results = ds.search(variable=["ci", "cl", "tas"])
/jobfs/162823973.gadi-pbs/ipykernel_3198199/1139839834.py:2: UserWarning: Value aliasing: variable='['ci', 'cl', 'tas']' → variable=['fld_s02i261','['ci', 'cl', 'tas']']
  results = ds.search(variable=["ci", "cl", "tas"])
/jobfs/162823973.gadi-pbs/ipykernel_3198199/1139839834.py:2: UserWarning: Value aliasing: variable='['ci', 'cl', 'tas']' → variable=['fld_s03i236','['ci', 'cl', 'tas']']
  results = ds.search(variable=["ci", "cl", "tas"])


,unique
filename,4740
path,4740
file_id,7
frequency,3
start_date,2760
end_date,2760
variable,276
variable_long_name,202
variable_standard_name,90
variable_cell_methods,5


<a name="regex-passthrough"></a>
## 9. Regex passthrough

Regex patterns and other non-string values bypass aliasing entirely and are
passed directly to the underlying catalog. This allows you to use the full power
of intake-esm's regex search when you need it.

In [17]:
# Regex string — passed through unchanged, no aliasing applied - so we don't expect ay results
results = ds.search(variable="ci|cl|tas")
results

,unique
filename,0
path,0
file_id,0
frequency,0
start_date,0
end_date,0
variable,0
variable_long_name,0
variable_standard_name,0
variable_cell_methods,0


<a name="warnings"></a>
## 10. Controlling alias warnings

Each alias expansion emits a `UserWarning`. You can suppress these using
Python's standard `warnings` module.

In [18]:
# Suppress UserWarnings from aliasing
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    results = cat.search(frequency="daily", realm="atmosphere")

results

,model,description,realm,frequency,variable
name,,,,,
HI_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP}",{atmos},{1day},"{fld_s30i202, fld_s03i236_min, fld_s08i234, fld_s02i201, fld_s02i208, fld_s30i301, fld_s02i207, fld_s02i204, fld_s03i807, fld_s03i225, fld_s03i245_min, fld_s05i214, fld_s03i227, fld_s03i317, press..."
HI_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP}",{atmos},{1day},"{fld_s30i202, fld_s03i236_min, fld_s30i206, fld_s08i234, fld_s02i201, fld_s02i208, fld_s30i301, fld_s02i207, fld_s02i204, fld_s03i807, fld_s03i225, fld_s03i245_min, fld_s03i227, fld_s03i317, press..."
HI_nl_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP, and land-use change disabled}",{atmos},{1day},"{fld_s30i202, fld_s03i236_min, fld_s30i206, fld_s08i234, fld_s02i201, fld_s02i208, fld_s30i301, fld_s02i207, fld_s02i204, fld_s03i807, fld_s03i225, fld_s03i245_min, fld_s05i214, fld_s03i227, fld_s..."
HI_noluc_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP, and land-use change disabled}",{atmos},{1day},"{fld_s30i202, fld_s03i236_min, fld_s30i206, fld_s08i234, fld_s02i201, fld_s02i208, fld_s30i301, fld_s02i207, fld_s02i204, fld_s03i807, fld_s03i225, fld_s03i245_min, fld_s03i227, fld_s03i317, press..."
PI_GWL_B2035,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2035 }",{atmos},{1day},"{fld_s03i236_min, fld_s02i208, fld_s03i807, fld_s03i225, fld_s03i245_min, fld_s03i227, fld_s03i236_max, fld_s08i223, fld_s03i802, fld_s03i810, fld_s03i226, fld_s16i222, fld_s03i811, fld_s03i801, f..."
PI_GWL_B2040,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2040}",{atmos},{1day},"{fld_s03i236_min, fld_s02i208, fld_s03i807, fld_s03i225, fld_s03i245_min, fld_s05i214, fld_s03i227, fld_s08i223, fld_s03i236_max, fld_s03i802, fld_s03i226, fld_s03i810, fld_s16i222, fld_s03i811, f..."
PI_GWL_B2045,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2045}",{atmos},{1day},"{fld_s03i236_min, fld_s02i208, fld_s03i807, fld_s03i225, fld_s03i245_min, fld_s03i227, fld_s08i223, fld_s03i236_max, fld_s03i810, fld_s03i802, fld_s03i226, fld_s16i222, fld_s03i811, fld_s03i801, f..."
PI_GWL_B2050,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2050}",{atmos},{1day},"{fld_s03i236_min, fld_s02i208, fld_s03i807, fld_s03i225, fld_s03i245_min, fld_s03i227, fld_s03i236_max, fld_s08i223, fld_s03i810, fld_s03i802, fld_s03i226, fld_s03i811, fld_s16i222, fld_s03i801, f..."
PI_GWL_B2055,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2055}",{atmos},{1day},"{fld_s03i236_min, fld_s02i208, fld_s03i807, fld_s03i225, fld_s03i245_min, fld_s05i214, fld_s03i227, fld_s03i236_max, fld_s08i223, fld_s03i802, fld_s03i810, fld_s03i226, fld_s16i222, fld_s03i811, f..."


<a name="unwrap"></a>
## 11. Escaping aliasing with `.unwrap()`

Both the top-level catalog wrapper and the per-dataset wrapper expose an
`.unwrap()` method that returns the raw underlying catalog object with no
aliasing layer. Use this when you need the exact type expected by another
library or want to call a method not supported by the wrapper.

In [19]:
# Get the raw DfFileCatalog (no aliasing wrapper)
raw_cat = cat.unwrap()
type(raw_cat)

intake_dataframe_catalog.core.DfFileCatalog

In [20]:
# Get the raw esm_datastore (no aliasing wrapper)
raw_ds = ds.unwrap()
type(raw_ds)

intake_esm.core.esm_datastore